# `Qwen3.6-35B-A6B-LoopHSSD` (v3) · Notebook 2: End-to-End Training & Whole-Model Pack Release

**Self-Resuming Multi-Session Pipeline (`Gate 0b → Gate 4`) + Low-Disk Streaming Whole-Model HF Release (Colab G4 · `RTX PRO 6000 Blackwell` `96 GiB` · `230 GB` SSD Safe)**

| Specialized Notebook | Purpose |
| :--- | :--- |
| [`colab_probe.ipynb`](colab_probe.ipynb) | **Zero-Training Diagnostic & Calibration Probes**: `Gate -1` + `Gate 0a` + `Gate 0p-1..3` + calibrated CLI overrides |
| **[`colab_train_e2e.ipynb`](colab_train_e2e.ipynb)** *(you are here)* | **End-to-End `Loop-HSSD v3` Training & Whole-Model Pack Release**: Resumable multi-stage evolution -> complete self-contained HF **Whole Model Pack** |
| [`colab_infer_eval.ipynb`](colab_infer_eval.ipynb) | **Shipped Whole-Model On-Weight Inference & Frontier `R=1/2/3` Eval**: `OptimizedLoopInferenceEngine`, `1M FP8` / `10M INT4` offload, frontier depth-curve suite |
| [`colab_train.ipynb`](colab_train.ipynb) | Baseline `64L` Track-A (`GDN-2` + `mHC`) single-notebook pipeline |

### Staged Kill-Gate Evolution (`TassP0lar/Qwen3.6-35B-A3B-Loop64` → Shipped Whole Model Pack)
1. **`Gate 0b` (`s1_loop_a6b.yaml`)**: `R=2 (64L, ~4.46B) → R=3 (88L A6B, ~5.97B)` Subspace-Aligned Sandwich Loop Heal (`ReEntryBridge` `r=256`, `w_anchor=0.85`, zero-init `PassExitGate`, task-driven `PassRouterAdapters`, per-pass Q/O rank-64 LoRA, Track-A `GDN-2`, frozen `32.338B` MoE SHA-256 verified).
2. **`Gate 1` (`s1b_hssd_distill.yaml`)**: Track-B **`HSSD v3`** Horizon-Split Surprise-Gated Delta Memory KL Distillation (`[513, 576]` disjoint SWA + `PeakBoltzmannGate` + 2D phase WY `chunk_kda` + smooth root-surprise `s = g^2` + `M=8` write-budgeted pages).
3. **`Gate 2a` (`s3_long_1m_lsa.yaml`)**: **`1M` GPU-Resident `FP8` SharedKV** + Per-Pass `D-LSA` (`num_query_passes=3`) + `ReRoPE` + Long-Prefix (`512K–992K` `no_grad`) / Short-Suffix (`32K` `grad`) read-side calibration.
4. **`Gate 3` (`s2_joint_bptt.yaml`)**: Joint `32K` High-Density CoT SFT + **`4×32K = 128K` Differentiable K/V-Replay Truncated BPTT** (`SegmentedKVBPTTRunner`, Top-`2,048` prior-segment K/V re-projection).
5. **`Gate 4` (`s4_rl.yaml`)**: Verifiable-Reward **GRPO / RSI** (`DAPO-Math-17k` + `KernelBench L1–L2`) with automatic held-out promotion/rollback gating.
6. **Final Release (`push_whole_model_pack(..., stream_low_disk=True)`)**: Folds optional `HotExpertDeltas`, serializes and streams sharded `model-00001-of-NNNNN.safetensors` **one `~4.5 GB` shard at a time** (`< 6 GB` peak temporary SSD!), and publishes `model.safetensors.index.json`, `config.json` (`auto_map` + `qloop_v3_arch`), standalone `configuration_qwen36_loophssd.py` & `modeling_qwen36_loophssd.py`, tokenizer, and Model Card to Hugging Face.

In [ ]:
REPO = "https://github.com/BaiTian6641/Qwen3.6-35B-A3B-Loop60.git"
BRANCH = "main"
%cd /content
!test -d /content/qloop/.git && test ! -d /content/qloop_repo && mv /content/qloop /content/qloop_repo || true
!test -d /content/qloop_repo || git clone -b $BRANCH $REPO /content/qloop_repo
%cd /content/qloop_repo
!git pull --ff-only
!bash scripts/colab_setup.sh

In [ ]:
import importlib
import os
import sys

if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
if "qloop" in sys.modules and getattr(sys.modules["qloop"], "__file__", None) is None:
    for _k in [k for k in sys.modules if k == "qloop" or k.startswith("qloop.")]:
        del sys.modules[_k]
importlib.invalidate_caches()

from qloop.colab import (
    hub_status,
    q,
    reclaim_disk,
    run_v3_e2e_and_ship_pack,
    setup_env,
    tail_metrics,
)

HF_SYNC_REPO = "TassP0lar/Qwen3.6-35B-A3B-Loop64"        # stage-by-stage commit-history sync repo
HF_PACK_REPO = "TassP0lar/Qwen3.6-35B-A6B-LoopHSSD-Pack" # final self-contained Whole Model Pack repo
DEBUG_LEVEL = 1                                          # 1=normal | 2=NaN/Inf module hooks | 3=CUDA_LAUNCH_BLOCKING
TIME_BUDGET_HOURS = 21.5                                 # clean save + HF commit before 24h Colab timeout
CALIBRATED_OVERRIDES = ()                                # paste `-s` overrides from colab_probe.ipynb if any

setup_env(hf_repo=HF_SYNC_REPO, run_root="/content/runs", hf_home="/content/hf_cache")
q(f"doctor --freeze requirements/colab.lock.txt --model Qwen/Qwen3.6-35B-A3B --debug {DEBUG_LEVEL}")
hub_status(HF_SYNC_REPO)

### 1 · Self-Resuming `Loop-HSSD v3` Pipeline + Streaming Whole-Model Pack Release
Pressing **`Runtime → Run all`** in any fresh Colab session queries `manifest.json` on `HF_SYNC_REPO`, skips completed stages, pulls only the active `~1–3 GB` trainable delta, trains within `TIME_BUDGET_HOURS`, prunes superseded local files (`>= 106 GiB` SSD free), and upon final stage completion streams the complete sharded **Whole Model Pack** to `HF_PACK_REPO`.

In [ ]:
V3_STAGES = (
    "s0_probe",
    "s1_loop_a6b",
    "s1b_hssd_distill",
    "s3_long_1m_lsa",
    "s2_joint_bptt",
    "s4_rl",
)

pipeline_result = run_v3_e2e_and_ship_pack(
    hf_repo=HF_SYNC_REPO,
    pack_repo=HF_PACK_REPO,
    stages=V3_STAGES,
    debug=DEBUG_LEVEL,
    time_budget_hours=TIME_BUDGET_HOURS,
    ship_whole_model_pack=True,
    stream_low_disk=True,
    merge_hot_experts=True,
    max_shard_size_bytes=4_500_000_000,
    overrides=CALIBRATED_OVERRIDES,
)
print("Pipeline & Release Result:", pipeline_result)
hub_status(HF_SYNC_REPO)

### 2 · Optional: Single-Stage Forensic Debug & Live Telemetry (`RUN_MANUAL_STAGE = True`)
Run or debug an individual `Loop-HSSD v3` stage (`s1_loop_a6b`, `s1b_hssd_distill`, `s3_long_1m_lsa`, `s2_joint_bptt`, `s4_rl`) with `--debug 2` (module-level NaN/Inf pinpointing + C++ traces) or `--debug 3` (`CUDA_LAUNCH_BLOCKING=1` + autograd anomaly + PyTorch memory snapshot).

In [ ]:
RUN_MANUAL_STAGE = False
MANUAL_STAGE = "s1_loop_a6b"  # s1_loop_a6b | s1b_hssd_distill | s3_long_1m_lsa | s2_joint_bptt | s4_rl
MANUAL_DEBUG = 2              # 2=NaN/Inf module hooks | 3=CUDA_LAUNCH_BLOCKING + anomaly + mem snapshot

if RUN_MANUAL_STAGE:
    q(
        f"train -c configs/stages/{MANUAL_STAGE}.yaml --debug {MANUAL_DEBUG} "
        f"-s checkpoint.sync.hf_repo={HF_SYNC_REPO}"
    )
    tail_metrics(MANUAL_STAGE)
    reclaim_disk(keep=MANUAL_STAGE)

### 3 · Optional: Manual Whole-Model Pack Export / Low-Disk Re-Stream (`RUN_MANUAL_PACK_PUSH = True`)
Directly invokes `push_whole_model_pack` (or local `export_whole_model_pack` on toy/custom checkpoints) to package any promoted stage (`s2_joint_bptt` or `s4_rl`) into a standalone multi-shard `safetensors` + custom HF modeling repository while keeping temporary SSD usage `< 6 GB`.

In [ ]:
RUN_MANUAL_PACK_PUSH = False
PACK_SOURCE_STAGE = "s4_rl"       # stage checkpoint to fold & ship (`s4_rl` or `s2_joint_bptt`)
PACK_SOURCE_CONFIG = "configs/stages/s2_joint_bptt.yaml"
EXPORT_LOCAL_DIR = ""             # optional local path for `export_whole_model_pack` when non-empty

if RUN_MANUAL_PACK_PUSH:
    from qloop.cli import _stack_with_ckpt
    from qloop.config import load_config
    from qloop.debug import crash_guard
    from qloop.export_pack import export_whole_model_pack, push_whole_model_pack

    reclaim_disk(keep=PACK_SOURCE_STAGE)
    with crash_guard("/content/runs/pack_export"):
        cfg = load_config(PACK_SOURCE_CONFIG, [f"checkpoint.sync.hf_repo={HF_SYNC_REPO}"])
        stack = _stack_with_ckpt(cfg, f"/content/runs/{PACK_SOURCE_STAGE}")
        if EXPORT_LOCAL_DIR:
            local_manifest = export_whole_model_pack(
                stack,
                EXPORT_LOCAL_DIR,
                tokenizer=stack.tokenizer,
                max_shard_size_bytes=4_500_000_000,
                merge_hot_experts=True,
            )
            print("Exported local Whole Model Pack:", local_manifest)
        push_manifest = push_whole_model_pack(
            stack,
            HF_PACK_REPO,
            token=os.environ.get("HF_TOKEN"),
            stream_low_disk=True,
            max_shard_size_bytes=4_500_000_000,
            merge_hot_experts=True,
        )
        print("Streamed Whole Model Pack to HF Hub:", push_manifest)